# Reproducing Silkuniene et al. (2025) from dandiset 001047

**Paper.** Silkuniene G, Silkunas M, Pakhomova ON, Semenov I, Pakhomov AG (2025). MHz compression of
pulse packets facilitates remote focusing of electroporation. *Bioelectrochemistry* 166, 109016.
[doi:10.1016/j.bioelechem.2025.109016](https://doi.org/10.1016/j.bioelechem.2025.109016). The claims
file was written from the NIH author manuscript in PubMed Central (PMC12331426): main text, Methods
and figure legends; there is no supplement, and the figures were not seen.

**Dandiset.** [DANDI:001047](https://dandiarchive.org/dandiset/001047/0.250701.1609), version
0.250701.1609. The paper cites no version (its data statement says "Data will be made available on
request"). This is the most recent published version. It has the same 88 assets and size as the draft
and as 0.241211.1856; the first version, 0.240610.2022, held 54 assets (the CT26 files; the CHO files
were added in December 2024). The pinned version is the one that names the paper (IsPublishedIn).

**What the dandiset holds.** Stitched single-channel images of cell monolayers after pulsed-field
exposure: CT26 cells at packet rates of 1 Hz (code 013) and 0.2 MHz (code 011), CANCAN and control
protocols, burst labels x1, x2, x3 and x5, YO-PRO-1 (FITC), Hoechst before exposure (DAPI) and
propidium (Cy3); and CHO cells, one burst label, YP and Hoechst. There are no images at 1 kHz or
10 kHz, no 300- or 900-ns pulses, no electrode elevations, no thermochromic film images and no CHO
series over burst numbers.

| Claim | Tested here |
|---|---|
| fig2-heating | Blocked: no film images |
| fig3a-centre-enhanced | Tested at the two rates present |
| fig3bc-rate-suppresses-off-target | Tested |
| fig3-lorentzian-profile | Tested at the two rates present |
| fig3-tail-deviation-falls-with-rate | Tested at the two rates present |
| fig4b-centre-no-rate-effect | Tested at the two rates present |
| fig4b-near-electrode-decline | Tested at the two rates present |
| fig4b-40-fold-ratio | Tested |
| fig4c-fwhm-falls-with-rate | Blocked: no 1 kHz or 10 kHz images (values at 1 Hz and 0.2 MHz shown) |
| fig4c-log-trend | Blocked: no 1 kHz or 10 kHz images |
| fig5-raised-electrodes-weaken-centre | Blocked: no elevation in the data or metadata |
| fig5ab-yp-spot-slightly-larger-than-pr | Tested, assuming electrodes touching the monolayer |
| fig5b-gaussian-pr-lorentzian-yp | Tested, assuming electrodes touching the monolayer |
| fig5c-nuclear-yp-centre | Blocked: no full-resolution post-exposure Hoechst images |
| fig6b, fig6c, fig6d (pulse duration) | Blocked: all files are 600-ns pulses |
| fig7be, fig7cf, fig7-sharpest-one-burst | Blocked: no CHO series over burst numbers (CT26 values by burst label shown) |

In [ ]:
import time
import warnings
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit

import cancan as cc

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()

version = cc.version_metadata()
assets = cc.list_assets()
with ProcessPoolExecutor(16) as ex:
    metas = list(ex.map(cc.read_metadata, assets))
files = []
for m in metas:
    p = cc.parse(m)
    files.append({**{k: p[k] for k in ("file", "cells", "channel", "phase", "pixel_um")}, "asset_id": m["asset_id"],
                  "rows": m["shape"][1], "cols": m["shape"][2], "resolution_attr": m["resolution_attr"],
                  "start": m["start"][:16], "areas": p["areas"], "desc": m["desc"], "size_mb": m["size"] / 1e6})
files = pd.DataFrame(files).sort_values("file").reset_index(drop=True)
print(f"dandiset {cc.DANDISET_ID} version {cc.DANDISET_VERSION}: {len(assets)} assets, "
      f"{sum(a['size'] for a in assets) / 1e9:.2f} GB; metadata read in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Title:", version["name"])
print("Description:", version["description"])
print("Species:", [s["name"] for s in version["assetsSummary"]["species"]], "; subjects:", version["assetsSummary"]["numberOfSubjects"])
print()
print("Files by cell line, channel and phase:")
print(files.groupby(["cells", "channel", "phase"]).size().to_string())
print("\nStated resolution attribute of the image data, all files:", sorted(files.resolution_attr.unique()))
print("CHO pixel size stated in the OME metadata (um):", sorted(files[files.cells == "CHO"].pixel_um.unique()))
print("\nImage shapes (rows x cols) and areas per image:")
files["n_areas"] = files.areas.map(len)
print(files.groupby(["cells", "rows", "cols", "n_areas", "start"]).size().to_string())
d0 = files[files.cells == "CT26"].desc.iloc[0]
d1 = files[files.cells == "CHO"].desc.iloc[0]
for d in (d0, d1):
    i = d.find("four")
    print("\nProtocol text in a", "CT26" if d is d0 else "CHO", "file:", d[i - 40:i + 420])

The dandiset's description and every file's protocol text say the electrodes were 3.8 mm apart and
the first pulse was 3.2 kV, with later pulses at 75, 50 and 25% of it. The paper used a 15.3-mm
array, a 7.2-kV first pulse and 16-pulse packets falling by 12.5% per phase (Methods 2.2). The CT26
images show the footprints of the paper's array, not a 3.8-mm one (checked below), so the protocol
text is wrong for them. Most CT26 session descriptions give "Unknown repetitions"; the burst number
is only in the subject label (x1 to x5), and one file says "repeated 5 times at 1Hz" for an x5
label. The `resolution` attribute of the image data is 96 in the CT26 files and 0.1 in the CHO files;
neither is a pixel size (the CHO OME metadata give 1.625 um), and the CT26 files state no pixel
size.

In [ ]:
area_rows = []
for _, f in files.iterrows():
    for a in f.areas:
        area_rows.append({"file": f.file, "cells": f.cells, "channel": f.channel, "phase": f.phase, **a})
areas_meta = pd.DataFrame(area_rows)
print("Exposure areas (one well each) by cell line, channel, rate code, protocol and burst label:")
print(areas_meta.groupby(["cells", "channel", "rate", "protocol", "label"]).size().unstack("channel").fillna(0).astype(int).to_string())
cho = files[files.cells == "CHO"]
print(f"\nCHO images: {len(cho)} files, {cho.rows.iloc[0]} x {cho.cols.iloc[0]} pixels at 1.625 um = "
      f"{cho.rows.iloc[0] * 1.625e-3:.1f} x {cho.cols.iloc[0] * 1.625e-3:.1f} mm; "
      f"wells: {areas_meta[areas_meta.cells == 'CHO'].subject.nunique()}")
print("CHO burst labels:", sorted(areas_meta[areas_meta.cells == "CHO"].label.unique()))

CHO: 17 wells (1 Hz CANCAN 6, 1 Hz control 5, 0.2 MHz CANCAN 6), each imaged before (Hoechst) and
after (YP), all with one burst label. The images span 13.0 x 13.8 mm, while the paper's electrode
footprints lie 6.4 mm from the array centre along each axis and reach 7.8 mm, so the CHO images
cannot show the array, and with one burst label they cannot hold the Figure 7 series. They are not
analysed further. The paper reports no CHO experiment at 1 Hz.

In [ ]:
ct = files[(files.cells == "CT26") & files.channel.isin(["FITC", "Cy3"])].reset_index(drop=True)
t0 = time.time()
loaded = cc.load_many([(r.asset_id, r.pixel_um) for r in ct.itertuples()])
print(f"streamed {len(ct)} CT26 images ({ct.size_mb.sum() / 1e3:.1f} GB) in {time.time() - t0:.0f} s")
imgs, recs = {}, []
for r, (img, (vmin, vmax)) in zip(ct.itertuples(), loaded):
    for a, (A, off) in zip(r.areas, cc.split_areas(img, len(r.areas))):
        c, how, score = cc.locate(A)
        key = (r.file, a["side"])
        imgs[key] = A
        recs.append({"file": r.file, "side": a["side"], "channel": r.channel, "rate": cc.RATE[a["rate"]],
                     "protocol": a["protocol"], "label": a["label"], "pixel_um": r.pixel_um, "raw_min": vmin,
                     "raw_max": vmax, "rows": A.shape[0], "cols": A.shape[1], "cy": c[0], "cx": c[1], "located_by": how,
                     "match": score})
areas = pd.DataFrame(recs)
print("raw pixel values (uint16, a.u.) over all CT26 images:", areas.raw_min.min(), "to", areas.raw_max.max())
print("\nAreas by channel, rate, protocol, burst label and how the array centre was found:")
print(areas.groupby(["channel", "rate", "protocol", "label", "located_by"]).size().unstack("located_by").fillna(0).astype(int).to_string())

Pixel size and array geometry. The footprint match uses the paper's geometry at the pixel sizes
taken in the helper (1.625 um for whole-well images, 3.25 um for two-well images). If those were
wrong, the best-matching electrode spacing would differ from the paper's. Below, the spacing is
searched from 5 to 8 mm on the areas where the footprints were found.

In [ ]:
rows = []
for r in areas[areas.located_by == "footprints"].itertuples():
    A = imgs[(r.file, r.side)]
    sc = [(cc.match_footprints(A, hs)[0], hs) for hs in np.arange(5.0, 8.01, 0.2)]
    rows.append({"file": r.file, "side": r.side, "pixel_um": r.pixel_um, "best_half_side_mm": max(sc)[1]})
geo = pd.DataFrame(rows)
print("Best-matching half side of the electrode square (mm), by pixel size taken; the paper's geometry gives",
      round(cc.HALF_SIDE_MM, 2))
print(geo.groupby("pixel_um").best_half_side_mm.describe().round(2).to_string())
w = files[(files.rows == 21870)]
print(f"\nWhole-well images: {w.rows.iloc[0]} x {w.cols.iloc[0]} pixels = "
      f"{w.rows.iloc[0] * 1.625e-3:.1f} x {w.cols.iloc[0] * 1.625e-3:.1f} mm at 1.625 um (a 6-well plate well is about 35 mm)")

In [ ]:
fig, axs = plt.subplots(9, 7, figsize=(21, 27))
o = cc.HALF_SIDE_MM / cc.PX_MM
for ax, r in zip(axs.flat, areas.itertuples()):
    A = imgs[(r.file, r.side)]
    lo, hi = np.percentile(A, [2, 99.5])
    ax.imshow(A, vmin=lo, vmax=hi, cmap="gray")
    for dy in (-o, o):
        for dx in (-o, o):
            ax.add_patch(plt.Circle((r.cx + dx, r.cy + dy), 1.4 / cc.PX_MM, fill=False, color="r" if r.located_by == "footprints" else "y"))
    ax.plot(r.cx, r.cy, "c+", ms=10)
    ax.set_title(f"{r.file[4:24]} {r.side} {r.channel}\n{r.rate} {r.protocol} {r.label} {r.located_by} {r.match:.3f}", fontsize=8)
    ax.axis("off")
for ax in axs.flat[len(areas):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

Each panel is one exposure area at 26-um pixels, scaled between its 2nd and 99.5th percentiles; the
circles are the four footprints placed at the array centre found (red: footprints matched, yellow:
centroid of the stained pattern). Where an area has almost no stain, neither method has anything to
find, and its profile is near zero whatever the centre.

Labels against the data. A single time-point image has no clock to check; what can be checked is
that the condition labels, and the left and right sides, match what the conditions do. The paper's
control has its lowest YP at the centre (Results 3.1), so the centre relative to 6 mm should be lower
in control areas than in CANCAN areas of the same rate and burst label.

In [ ]:
P, PC = cc.POS_WHOLE_MM, cc.POS_CONTIG_MM
prof = {}
for i, r in areas.iterrows():
    A = imgs[(r.file, r.side)]
    c = (r.cy, r.cx)
    bg, bgmin = cc.background(A, c)
    areas.loc[i, "bg"], areas.loc[i, "bg_min"] = bg, bgmin
    prof[(r.file, r.side)] = {"diag_whole": cc.profiles(A, c, P, cc.DIAGONALS), "diag_contig": cc.profiles(A, c, PC, cc.DIAGONALS),
                              "par_whole": cc.profiles(A, c, P, cc.PARALLEL), "par_contig": cc.profiles(A, c, PC, cc.PARALLEL)}
areas["usable"] = [not np.isnan(prof[(r.file, r.side)]["diag_whole"]).any() and not np.isnan(prof[(r.file, r.side)]["par_whole"]).any()
                   for r in areas.itertuples()]
print("Areas dropped because an ROI line leaves the image (centre placed near an edge):", int((~areas.usable).sum()), "of", len(areas))
print(areas[~areas.usable][["file", "side", "channel", "rate", "protocol", "label", "located_by", "cy", "cx"]].to_string(index=False))

def area_profile(key, kind="diag", o1="a", o2="a", bgkind="bg"):
    pos = P if o1 == "a" else PC
    raw = prof[key][f"{kind}_{'whole' if o1 == 'a' else 'contig'}"]
    bg = areas.set_index(["file", "side"]).loc[key, bgkind]
    return cc.pool(raw - bg, pos, o2)

def at(pos, y, x):
    return y[np.argmin(np.abs(pos - x))]

fitc = areas[(areas.channel == "FITC") & areas.usable].copy()
for i, r in fitc.iterrows():
    pos, y = area_profile((r.file, r.side))
    fitc.loc[i, "y0"], fitc.loc[i, "y6"], fitc.loc[i, "y7"] = at(pos, y, 0), at(pos, y, 6), at(pos, y, 7)
print("\nBackground-subtracted YP (a.u.) at 0, 6 and 7 mm, and background, by group (mean over areas):")
print(fitc.groupby(["label", "rate", "protocol"])[["y0", "y6", "y7", "bg"]].agg(["mean", "count"]).round(1).to_string())
fitc["centre_minus_6mm"] = fitc.y0 - fitc.y6
print("\nCentre minus 6 mm, CANCAN against control in the same image (paired areas):")
pairs = fitc.groupby("file").filter(lambda g: len(g) == 2 and set(g.protocol) == {"CANCAN", "control"})
pp = pairs.pivot_table(index="file", columns="protocol", values="centre_minus_6mm")
print(pp.round(1).to_string())
print("CANCAN above control in", int((pp.CANCAN > pp.control).sum()), "of", len(pp), "images")

Background-subtracted YP is in the camera's arbitrary units, as in the paper. Whole-well images
(taken on May 23 at 14:13) have backgrounds several times those of the two-well images, so their
a.u. are not on the same scale as the others; they are kept, as each is one well.

Duplicates. The protocol asks for recordings deposited more than once. Every pair of exposure areas
with the same channel and condition was compared by the peak of the normalised cross-correlation
of their high-passed images (16 x 16 mm around the array centre), under translation, flips and a
half turn (a well imaged twice would correlate at the scale of single cells).

In [ ]:
from numpy.fft import fft2, ifft2
from scipy.ndimage import gaussian_filter

def hp(A):
    B = np.log1p(np.clip(A - np.percentile(A, 5), 0, None))
    B = gaussian_filter(B - gaussian_filter(B, 8), 1)
    return (B - B.mean()) / B.std()

def xcorr(A, B):
    H, W = max(A.shape[0], B.shape[0]), max(A.shape[1], B.shape[1])
    a, b = np.zeros((2 * H, 2 * W)), np.zeros((2 * H, 2 * W))
    a[:A.shape[0], :A.shape[1]], b[:B.shape[0], :B.shape[1]] = hp(A), hp(B)
    return float((np.real(ifft2(fft2(a) * np.conj(fft2(b)))) / np.sqrt(A.size * B.size)).max())

res = []
def crop(r):
    A = imgs[(r.file, r.side)]
    h = int(8 / cc.PX_MM)
    cy, cx = int(r.cy), int(r.cx)
    return A[max(cy - h, 0):cy + h, max(cx - h, 0):cx + h]
for _, g in areas.groupby(["channel", "rate", "protocol", "label"]):
    g = list(g.itertuples())
    for i in range(len(g)):
        for j in range(i + 1, len(g)):
            A, B = crop(g[i]), crop(g[j])
            res.append((max(xcorr(A, Bf) for Bf in (B, B[::-1], B[:, ::-1], B[::-1, ::-1])), g[i], g[j]))
res.sort(key=lambda t: -t[0])
print("pairs compared:", len(res))
print("highest correlations:")
for v, a, b in res[:5]:
    print(round(v, 3), a.file, a.side, "|", b.file, b.side)
# positive control: an area against itself shifted by 3 pixels with independent noise
A = crop(next(areas.itertuples()))
noisy = np.roll(A, 3, axis=1) + np.random.default_rng(0).normal(0, A.std() * 0.5, A.shape)
print("positive control, one area against a shifted, noisy copy of itself:", round(xcorr(A, noisy), 3))

## Methods common to the claims

Every profile is the background-subtracted mean in 406 x 325 um ROIs along the diagonals (F1, F2),
with the readings of the claims file: ROIs at whole-mm distances (O1 a) or contiguous from the
centre (O1 b), and the two diagonals averaged at each signed position (O2 a) or the four
half-diagonals at each distance (O2 b). Groups are means +/- s.e. over exposure areas (F3).

Burst number. The paper's rate experiments (set S1) used 2 bursts (x4). The dataset gives burst
numbers only as labels x1, x2, x3 and x5, and the two rates share only x1 and x5. Whether x1 is one
burst or one repetition after the first cannot be told from the files. The rate claims are therefore
tested at both shared labels, x1 and x5; an assumption, stated in the record. A claim is taken as
reproduced when it holds at both, partially when at one only. The controls are those of the same
label at both rates (the paper's control rate is not given; per-rate controls are shown as a
variant).

In [ ]:
READINGS = [(o1, o2) for o1 in "ab" for o2 in "ab"]

def group_profile(sel, kind="diag", o1="a", o2="a", bgkind="bg"):
    ys = [area_profile((r.file, r.side), kind, o1, o2, bgkind)[1] for r in sel.itertuples()]
    pos = P if o1 == "a" else PC
    ys = np.array(ys)
    return pos, np.nanmean(ys, 0), np.nanstd(ys, 0, ddof=1) / np.sqrt(np.isfinite(ys).sum(0)), ys

def sel(rate=None, protocol="CANCAN", label="x1", channel="FITC"):
    s = areas[(areas.channel == channel) & (areas.label == label) & areas.usable]
    if protocol:
        s = s[s.protocol == protocol]
    if rate:
        s = s[s.rate == rate]
    return s

RATES = ["1 Hz", "0.2 MHz"]
LABELS = ["x1", "x5"]
for lab in LABELS:
    for rate in RATES:
        pos, m, se, ys = group_profile(sel(rate, "CANCAN", lab))
        print(f"{lab} {rate:8s} CANCAN n={len(ys)}  mean YP (a.u.) at -7..7 mm:", np.round(m).astype(int).tolist())
    pos, m, se, ys = group_profile(sel(None, "control", lab))
    print(f"{lab} control  n={len(ys)}          mean YP (a.u.) at -7..7 mm:", np.round(m).astype(int).tolist())
    print(f"{' ' * 33}s.e.:", np.round(se).astype(int).tolist())

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, lab in zip(axs, LABELS):
    for rate, col in zip(RATES, ["C0", "C3"]):
        pos, m, se, ys = group_profile(sel(rate, "CANCAN", lab))
        ax.errorbar(pos, m, se, color=col, marker="o", label=f"CANCAN {rate} (n={len(ys)})")
    pos, m, se, ys = group_profile(sel(None, "control", lab))
    ax.errorbar(pos, m, se, color="k", marker="s", ls="--", label=f"control (n={len(ys)})")
    ax.set_title(f"burst label {lab}, diagonal profile (O1 a, O2 a)")
    ax.set_xlabel("distance from centre (mm)")
    ax.set_ylabel("YP, background subtracted (a.u.)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## fig3a-centre-enhanced

> Every CANCAN protocol enhanced electroporation at the array centre, while in the control (single
> unipolar pulse per packet) YP uptake decreased gradually toward the centre.

Criterion: for each CANCAN rate the mean diagonal profile has its maximum at x = 0; for the control,
the mean at 0 is below the means at 6 and 7 mm by a gap larger than the error bars (F8). Only 1 Hz
and 0.2 MHz are in the dataset.

In [ ]:
rows = []
for lab in LABELS:
    for o1, o2 in READINGS:
        for rate in RATES:
            pos, m, se, ys = group_profile(sel(rate, "CANCAN", lab), o1=o1, o2=o2)
            rows.append({"label": lab, "O1": o1, "O2": o2, "group": f"CANCAN {rate}", "n": len(ys),
                         "argmax_mm": round(float(pos[np.nanargmax(m)]), 2), "y0": m[np.argmin(np.abs(pos))],
                         "max": np.nanmax(m), "holds": bool(abs(pos[np.nanargmax(m)]) < 1e-9)})
        pos, m, se, ys = group_profile(sel(None, "control", lab), o1=o1, o2=o2)
        i0, i6, i7 = (np.argmin(np.abs(pos - x)) for x in (0, 6, 7))
        g6 = cc.gap_significant(m[i6], se[i6], m[i0], se[i0])
        g7 = cc.gap_significant(m[i7], se[i7], m[i0], se[i0])
        rows.append({"label": lab, "O1": o1, "O2": o2, "group": "control", "n": len(ys), "y0": m[i0], "y6": m[i6],
                     "y7": m[i7], "se0": se[i0], "se6": se[i6], "se7": se[i7],
                     "holds": bool(m[i0] < m[i6] and m[i0] < m[i7] and g6[0] and g7[0])})
t3a = pd.DataFrame(rows)
print(t3a.round(1).to_string(index=False))
summ = t3a.groupby("label").holds.all()
print("\nholds for every rate and the control under every reading, by label:", summ.to_dict())
for lab in LABELS:
    for rate in RATES:
        pos, m, se, ys = group_profile(sel(rate, "control", lab))
        print(f"variant, control of {rate} only, {lab}: n={len(ys)}, y0={m[7]:.0f}+/-{se[7]:.0f}, y6={m[13]:.0f}+/-{se[13]:.0f}, y7={m[14]:.0f}+/-{se[14]:.0f}")

**Result: not reproduced.** At neither shared burst label does a CANCAN profile peak at the centre under every reading: at x1 the 0.2 MHz CANCAN wells have essentially no stain (centre -1.4 a.u., maximum 32 a.u.) and the 1 Hz CANCAN profile peaks at 6.5 to 7 mm; at x5 the 0.2 MHz profile peaks within 1 mm of the centre (at -1, -0.4 or 1 mm, not at 0) and the 1 Hz profile peaks off-centre. The pooled control is lower at the centre than at 6 and 7 mm at x1 (245 against 868 and 618 a.u., O1 a, O2 a), but the gap does not exceed the error bars; at x5 it is not lower at 7 mm. Only 1 Hz and 0.2 MHz are in the dataset.

## fig3bc-rate-suppresses-off-target

> Raising the packet rate from 1 Hz to 0.2 MHz suppressed electroporation better both near the
> electrodes (diagonal profile) and at the perimeter between electrodes (profile parallel to the sides).

Criterion: mean YP at 0.2 MHz below 1 Hz by a gap larger than the error bars at 6 mm, at 7 mm (diagonal)
and at the perimeter (parallel profile: (a) outermost ROIs, (b) mean of ROIs at 4 mm or more); two-sided
t tests reported. The parallel profile is measured on every area (the paper measured it in one set).

In [ ]:
def per_area(selection, kind, o1, o2, xs=None, beyond=None):
    out = []
    for r in selection.itertuples():
        pos, y = area_profile((r.file, r.side), kind, o1, o2)
        if beyond is not None:
            out.append(np.nanmean(y[np.abs(pos) >= beyond]))
        else:
            out.append(np.nanmean([at(pos, y, x) for x in xs]))
    return np.array(out)

rows = []
for lab in LABELS:
    for o1, o2 in READINGS:
        outer = P.max() if o1 == "a" else PC.max()
        tests = {"6 mm": ("diag", dict(xs=[6, -6] if o2 == "a" else [6])), "7 mm": ("diag", dict(xs=[7, -7] if o2 == "a" else [7])),
                 "perimeter (a) outermost": ("par", dict(xs=[outer, -outer])), "perimeter (b) >= 4 mm": ("par", dict(beyond=4))}
        for name, (kind, kw) in tests.items():
            hi = per_area(sel("1 Hz", "CANCAN", lab), kind, o1, o2, **kw)
            lo = per_area(sel("0.2 MHz", "CANCAN", lab), kind, o1, o2, **kw)
            m1, s1 = hi.mean(), hi.std(ddof=1) / np.sqrt(len(hi))
            m2, s2 = lo.mean(), lo.std(ddof=1) / np.sqrt(len(lo))
            g = cc.gap_significant(m1, s1, m2, s2)
            rows.append({"label": lab, "O1": o1, "O2": o2, "where": name, "1 Hz": m1, "se_1Hz": s1, "0.2 MHz": m2, "se_02": s2,
                         "ratio": m1 / m2 if m2 != 0 else np.nan, "gap": g[1], "holds": bool(m2 < m1 and g[0]),
                         "t_p": stats.ttest_ind(hi, lo).pvalue, "n": f"{len(hi)}/{len(lo)}"})
t3bc = pd.DataFrame(rows)
print(t3bc.round(3).to_string(index=False))
print("\nholds at 6 mm, 7 mm and the perimeter under every reading, by label:", t3bc.groupby("label").holds.all().to_dict())
print("near electrodes only (6 and 7 mm), by label:", t3bc[t3bc["where"].isin(["6 mm", "7 mm"])].groupby("label").holds.all().to_dict())

**Result: not reproduced.** At burst label x1, YP is lower at 0.2 MHz than at 1 Hz at 6 mm (9 against 153 a.u.) and 7 mm (5 against 174), with a gap beyond the error bars at 6 mm only; at the perimeter the outermost parallel ROIs are lower at 0.2 MHz, beyond the error bars under O1 a only, and the mean beyond 4 mm is not (7 against -10 a.u.). No t test reaches p < 0.05 (smallest p = 0.16). At x5, 0.2 MHz is higher than 1 Hz at 6 and 7 mm (1534 against 1173 and 917 against 780 a.u.) and lower at the perimeter without a gap. The direction near the electrodes matches the paper at x1, not at x5.

## fig3-lorentzian-profile

> The decline of YP emission from the centre was bell-shaped and fitted a Lorentzian function over -4
> to 4 mm nearly perfectly (R > 0.9, p < 0.001) for every packet rate.

Criterion: the Lorentzian (F4) fitted to the mean diagonal profile, and to the parallel profile,
over -4 to 4 mm for each rate; R is the Pearson correlation of fitted and measured values and p its
test. Reproduced when R > 0.9 with p < 0.001 for every rate and profile under every reading.

In [ ]:
fits = {}
rows = []
for lab in LABELS:
    for rate in RATES:
        for kind in ("diag", "par"):
            for o1, o2 in READINGS:
                pos, m, se, ys = group_profile(sel(rate, "CANCAN", lab), kind, o1, o2)
                f = cc.fit(cc.lorentz, pos, m, 4)
                fits[(lab, rate, kind, o1, o2)] = (pos, m, f)
                rows.append({"label": lab, "rate": rate, "profile": kind, "O1": o1, "O2": o2, "a": f["a"] if f else np.nan,
                             "FWHM_mm": f["w"] if f else np.nan, "R": f["R"] if f else np.nan, "p": f["p"] if f else np.nan,
                             "points": f["n"] if f else 0, "holds": bool(f and f["R"] > 0.9 and f["p"] < 0.001)})
tl = pd.DataFrame(rows)
print(tl.round(4).to_string(index=False))
print("\nholds for both rates, both profiles and every reading, by label:", tl.groupby("label").holds.all().to_dict())
print("fits that failed:", int(tl.a.isna().sum()))

**Result: not reproduced.** No rate passes at either label under every reading. The 1 Hz CANCAN fits have R from -0.89 to 0.72; the x1 0.2 MHz profiles have no central peak (R negative, FWHM at the 100-mm bound). Only the x5 0.2 MHz profiles come close, R 0.80 to 0.96 with p below 0.01, passing under 4 of 8 profile and reading combinations, with FWHM 7.0 to 8.9 mm.

## fig3-tail-deviation-falls-with-rate

> Beyond the fitted range, the data deviated from the Lorentzian fit less as the packet rate increased.

Criterion: fits of the previous claim extended to all distances; the deviation is the mean of
(data - fit) over ROIs beyond 4 mm, in (a) a.u. and (b) percent of the fitted peak a. Reproduced when
the deviation is largest at 1 Hz and smallest at 0.2 MHz, and its Spearman correlation with rate is
negative. With two rates the Spearman coefficient can only be +1 or -1 and has no test.

In [ ]:
rows = []
for lab in LABELS:
    for kind in ("diag", "par"):
        for o1, o2 in READINGS:
            dev = {}
            for rate in RATES:
                pos, m, f = fits[(lab, rate, kind, o1, o2)]
                far = np.abs(pos) > 4
                d = np.nanmean(m[far] - cc.lorentz(pos[far], f["a"], f["w"])) if f else np.nan
                dev[rate] = (d, 100 * d / f["a"] if f else np.nan)
            for u, k in (("a.u.", 0), ("% of a", 1)):
                v = [dev[r][k] for r in RATES]
                rho = stats.spearmanr([1, 208333], v)[0] if np.all(np.isfinite(v)) else np.nan
                rows.append({"label": lab, "profile": kind, "O1": o1, "O2": o2, "unit": u, "dev_1Hz": v[0], "dev_0.2MHz": v[1],
                             "spearman": rho, "holds": bool(v[0] > v[1] and rho < 0)})
tt = pd.DataFrame(rows)
print(tt.round(2).to_string(index=False))
print("\nholds under every reading, diagonal profile, by label:", tt[tt.profile == "diag"].groupby("label").holds.all().to_dict())
print("holds under every reading, both profiles, by label:", tt.groupby("label").holds.all().to_dict())

**Result: partially reproduced.** At burst label x1 the deviation beyond 4 mm is larger at 1 Hz than at 0.2 MHz under every reading (diagonal 145 against -4 a.u. under O1 a), so the Spearman coefficient over the two rates is -1; but the x1 0.2 MHz profile is flat near zero and its fit has no peak, so this says only that there is less stain far out. At x5 the order is reversed under every reading (-207 against 430 a.u.). With two rates, the trend over rates cannot be tested.

## fig4b-centre-no-rate-effect

> YP uptake at the array centre showed no significant effect of packet rate.

A claim of absence, judged on the paper's test: two-sided Student's t tests of per-area YP at x = 0,
Bonferroni-corrected. With two rates there is one comparison under both readings of the claims file;
the paper's correction factors for its five rates (4 against 1 Hz, 10 for all pairs) are shown as
variants.

In [ ]:
rows = []
for lab in LABELS:
    for o2 in "ab":
        v = {}
        for rate in RATES:
            v[rate] = per_area(sel(rate, "CANCAN", lab), "diag", "a", o2, xs=[0])
        t = stats.ttest_ind(v["1 Hz"], v["0.2 MHz"])
        rows.append({"label": lab, "O2": o2, "mean_1Hz": v["1 Hz"].mean(), "se_1Hz": v["1 Hz"].std(ddof=1) / np.sqrt(len(v["1 Hz"])),
                     "mean_0.2MHz": v["0.2 MHz"].mean(), "se_0.2MHz": v["0.2 MHz"].std(ddof=1) / np.sqrt(len(v["0.2 MHz"])),
                     "ratio_02_over_1Hz": v["0.2 MHz"].mean() / v["1 Hz"].mean(), "t": t.statistic, "p (1 comparison)": t.pvalue,
                     "p x4": min(1, 4 * t.pvalue), "p x10": min(1, 10 * t.pvalue), "n": f"{len(v['1 Hz'])}/{len(v['0.2 MHz'])}"})
tc = pd.DataFrame(rows)
print(tc.round(4).to_string(index=False))
print("\nno significant difference (corrected p >= 0.05), by label:", (tc.groupby("label")["p (1 comparison)"].min() >= 0.05).to_dict())

**Result: reproduced.** The centre YP does not differ between 1 Hz and 0.2 MHz at either label (x1: 1.0 against -1.4 a.u., p = 0.93; x5: 1576 against 2275 a.u., ratio 1.44, p = 0.54), uncorrected and so under any Bonferroni factor. At x1 both rates leave the centre unstained, which is not the paper's situation of a stained centre at every rate.

## fig4b-near-electrode-decline

> In contrast to the centre, YP uptake at 6 and 7 mm from the centre (1.65 and 0.65 mm from the
> electrodes) declined steeply as the packet rate increased.

Criterion: a power function YP = c rate^k fitted at 6 and at 7 mm, to (a) group means or (b)
per-area values; reproduced when k < 0 at both distances and the 0.2 MHz mean is below 1 Hz with a gap
larger than the error bars at both. With two rates, the power fit to the means passes through both.

In [ ]:
rates_hz = {"1 Hz": 1.0, "0.2 MHz": 208333.0}
def power(r, c, k):
    return c * r ** k
rows = []
for lab in LABELS:
    for o2 in "ab":
        for x in (6, 7):
            v = {rate: per_area(sel(rate, "CANCAN", lab), "diag", "a", o2, xs=[x, -x] if o2 == "a" else [x]) for rate in RATES}
            xs_m = np.array([rates_hz[r] for r in RATES]); ys_m = np.array([v[r].mean() for r in RATES])
            xs_e = np.concatenate([[rates_hz[r]] * len(v[r]) for r in RATES]); ys_e = np.concatenate([v[r] for r in RATES])
            ks = {}
            for nm, xx, yy in (("means", xs_m, ys_m), ("per-area", xs_e, ys_e)):
                try:
                    p_, _ = curve_fit(power, xx, yy, p0=[max(yy[0], 1), -0.1], maxfev=20000)
                    ks[nm] = p_[1]
                except RuntimeError:
                    ks[nm] = np.nan
            s = {r: v[r].std(ddof=1) / np.sqrt(len(v[r])) for r in RATES}
            g = cc.gap_significant(ys_m[0], s["1 Hz"], ys_m[1], s["0.2 MHz"])
            rows.append({"label": lab, "O2": o2, "x_mm": x, "mean_1Hz": ys_m[0], "se_1Hz": s["1 Hz"], "mean_0.2MHz": ys_m[1],
                         "se_0.2MHz": s["0.2 MHz"], "k_means": ks["means"], "k_per_area": ks["per-area"], "gap": g[1],
                         "t_p": stats.ttest_ind(v["1 Hz"], v["0.2 MHz"]).pvalue,
                         "holds": bool(ks["means"] < 0 and ks["per-area"] < 0 and ys_m[1] < ys_m[0] and g[0])})
tn = pd.DataFrame(rows)
print(tn.round(4).to_string(index=False))
print("\nholds at 6 and 7 mm under every reading, by label:", tn.groupby("label").holds.all().to_dict())

**Result: not reproduced.** At label x1 the power exponent is negative at 6 and 7 mm (k = -0.23 and -0.29) and the 0.2 MHz means are lower (9 against 153 and 5 against 174 a.u.), with a gap beyond the error bars at 6 mm but not at 7 mm (t test p = 0.17 and 0.33). At x5 the exponent is positive at both distances (0.022 and 0.013): YP near the electrodes is not lower at 0.2 MHz.

## fig4b-40-fold-ratio

> At 0.2 MHz, YP uptake at the centre was 40 times that near the electrodes, despite the field there
> being about 4 times weaker (3.3 against about 14 kV/cm).

Criterion: the ratio of group-mean background-subtracted YP at x = 0 to that near the electrodes (O5:
6 or 7 mm), under O2 a or b; reproduced when within 36 to 44 under one reading. The field values come
from the paper's simulation and are not tested.

In [ ]:
rows = []
for lab in LABELS:
    for o1, o2 in READINGS:
        pos, m, se, ys = group_profile(sel("0.2 MHz", "CANCAN", lab), "diag", o1, o2)
        y0 = at(pos, m, 0)
        for near in (6, 7):
            yn = at(pos, m, near) if o2 == "b" else np.mean([at(pos, m, near), at(pos, m, -near)])
            rows.append({"label": lab, "O1": o1, "O2": o2, "near_mm": near, "y0": y0, "y_near": yn,
                         "ratio": y0 / yn if yn > 0 else np.nan, "n": len(ys)})
t40 = pd.DataFrame(rows)
print(t40.round(2).to_string(index=False))
print("\nany reading within 36 to 44, by label:", t40.groupby("label").ratio.apply(lambda r: bool(((r >= 36) & (r <= 44)).any())).to_dict())
print("range of ratios by label:", t40.groupby("label").ratio.agg(["min", "max"]).round(2).to_dict("index"))
print("\nvariant, background from the footprint with the lowest mean (O1 a, O2 a):")
for lab in LABELS:
    pos, m, se, ys = group_profile(sel("0.2 MHz", "CANCAN", lab), "diag", "a", "a", bgkind="bg_min")
    print(f"  {lab}: y0={at(pos, m, 0):.1f}, y6={np.mean([at(pos, m, 6), at(pos, m, -6)]):.1f}, y7={np.mean([at(pos, m, 7), at(pos, m, -7)]):.1f}")

**Result: not reproduced.** The ratio of centre to near-electrode YP at 0.2 MHz is far from 40 under every reading: at x1 the centre is below background (-1.4 a.u.) and the ratio is negative (-0.29 to -0.15); at x5 it is 1.44 to 2.48. With the background from the darkest footprint, x1 gives 11.8 against 22.4 and 18.2 a.u., still no central excess.

## fig4c-fwhm-falls-with-rate and fig4c-log-trend (blocked)

Both rest on the 1 kHz and 10 kHz groups, which the dataset does not hold. For the record, the
per-area Lorentzian FWHM (fit over -4 to 4 mm, O1 a, O2 a) at the two rates present is printed; it
decides no outcome.

In [ ]:
rows = []
for lab in LABELS:
    for rate in RATES:
        for r in sel(rate, "CANCAN", lab).itertuples():
            pos, y = area_profile((r.file, r.side))
            f = cc.fit(cc.lorentz, pos, y, 4)
            rows.append({"label": lab, "rate": rate, "file": r.file, "side": r.side, "FWHM_mm": f["w"] if f else np.nan,
                         "R": f["R"] if f else np.nan, "at_bound": bool(f and (f["w"] > 99 or f["w"] < 0.01))})
fw = pd.DataFrame(rows)
print(fw.round(3).to_string(index=False))
print(fw.groupby(["label", "rate"]).agg(mean_FWHM=("FWHM_mm", "mean"), n=("FWHM_mm", "size"), at_bound=("at_bound", "sum")).round(2).to_string())

## fig5ab-yp-spot-slightly-larger-than-pr

> The central spot of YP staining (electroporation) was only slightly larger than the spot of Pr
> staining (cell killing).

Set S2 is 0.2 MHz, 2 bursts (x4), electrodes touching the monolayer or raised. The dataset gives no
elevation; its 0.2 MHz CANCAN wells imaged for both YP and Pr (May 24) are taken as touching the
monolayer (Methods 2.2, the array "positioned ... until touching the bottom of the culture dish"), an
assumption. They carry burst labels x1, x3 and x5, two wells each; the claim is judged on the six
pooled, with each label shown. Criterion: FWHM of the Lorentzian fit to mean YP and of the Gaussian
fit to mean Pr over -5 to 5 mm (FWHM = 2.4 sigma; 2.355 sigma as a variant); reproduced when YP FWHM /
Pr FWHM lies in (a) 1 to 1.25 and (b) 1 to 1.5. Pr areas are located in their own images.

In [ ]:
cy3_files = set(areas[areas.channel == "Cy3"].file)
s2_cy3 = areas[(areas.channel == "Cy3") & (areas.rate == "0.2 MHz") & (areas.protocol == "CANCAN") & areas.usable]
# the YP image of the same plate pair: same subjects on the same sides and the same image width, imaged on May 24
def yp_partner(r):
    f = files.set_index("file")
    subj = [a["subject"] for a in f.loc[r.file, "areas"]]
    cand = files[(files.channel == "FITC") & (files.start.str.startswith("2024-05-24"))]
    for c in cand.itertuples():
        if [a["subject"] for a in c.areas] == subj and c.cols == f.loc[r.file, "cols"]:
            return c.file
s2 = s2_cy3.assign(yp_file=[yp_partner(r) for r in s2_cy3.itertuples()])
print(s2[["file", "side", "label", "located_by", "yp_file"]].to_string(index=False))
s2_yp = pd.concat([areas[(areas.file == r.yp_file) & (areas.side == r.side) & areas.usable] for r in s2.itertuples()])
print("YP partners found:", len(s2_yp), "of", len(s2))

rows = []
groups = {"pooled": None, "x1": "x1", "x3": "x3", "x5": "x5"}
s2fits = {}
for gname, lab in groups.items():
    py = s2_yp if lab is None else s2_yp[s2_yp.label == lab]
    pr = s2 if lab is None else s2[s2.label == lab]
    for o1, o2 in READINGS:
        pos, my, sy, _ = group_profile(py, "diag", o1, o2)
        pos2, mp, sp, _ = group_profile(pr, "diag", o1, o2)
        fy = cc.fit(cc.lorentz, pos, my, 5)
        fp = cc.fit(cc.gauss, pos2, mp, 5)
        s2fits[(gname, o1, o2)] = (pos, my, pos2, mp)
        ry = fy["w"] if fy else np.nan
        rp = 2.4 * fp["w"] if fp else np.nan
        rows.append({"group": gname, "O1": o1, "O2": o2, "n_YP": len(py), "n_Pr": len(pr), "YP_FWHM": ry, "Pr_FWHM_2.4s": rp,
                     "ratio": ry / rp, "ratio_2.355s": ry / (2.355 * fp["w"]) if fp else np.nan,
                     "in_1_1.25": bool(1 <= ry / rp <= 1.25), "in_1_1.5": bool(1 <= ry / rp <= 1.5)})
t5 = pd.DataFrame(rows)
print(t5.round(3).to_string(index=False))
pooled = t5[t5.group == "pooled"]
print("\npooled: within 1 to 1.25 under every reading:", bool(pooled["in_1_1.25"].all()), "; within 1 to 1.5 under every reading:", bool(pooled["in_1_1.5"].all()))

**Result: not reproduced.** Over the five 0.2 MHz CANCAN wells with both YP and Pr images, the YP spot is narrower than the Pr spot under every reading: YP FWHM 6.8 to 8.0 mm against Pr FWHM 10.4 to 10.5 mm, ratio 0.65 to 0.76 (0.67 to 0.78 with 2.355 sigma), outside both 1 to 1.25 and 1 to 1.5. The x3 and x5 wells give 0.52 to 0.69; the single x1 well has no peak to fit. These FWHM values, 4.8 to 13.7 mm, are far wider than the paper's (1.1 to 4 mm). This rests on the assumption that these wells had the electrodes touching the monolayer.

## fig5b-gaussian-pr-lorentzian-yp

> Pr intensity declined from the peak as a Gaussian and YP as a Lorentzian; for the 0-mm data the
> R2 was 0.92 (Lorentzian) against 0.96 (Gaussian) for Pr, and 0.98 against 0.95 for YP, fitted from
> -5 to 5 mm with p < 0.0001.

Same wells and assumption as the previous claim. R2 = 1 - SSres / SStot over the fitted points, on
the mean profiles in percent of their 0-mm value (the fits are unchanged by scale). Reproduced when
every R2 is within tolerance (0.92 +/- 0.092, 0.96 +/- 0.096, 0.98 +/- 0.098, 0.95 +/- 0.095) and both
orderings hold under every reading.

In [ ]:
rows = []
for (gname, o1, o2), (pos, my, pos2, mp) in s2fits.items():
    my_pc, mp_pc = 100 * my / at(pos, my, 0), 100 * mp / at(pos2, mp, 0)
    r = {"group": gname, "O1": o1, "O2": o2}
    for nm, x, y in (("Pr", pos2, mp_pc), ("YP", pos, my_pc)):
        for fn, fun in (("L", cc.lorentz), ("G", cc.gauss)):
            f = cc.fit(fun, x, y, 5)
            r[f"{nm}_{fn}_R2"] = f["R2"] if f else np.nan
            r[f"{nm}_{fn}_p"] = f["p"] if f else np.nan
    r["within_tol"] = bool(abs(r["Pr_L_R2"] - 0.92) <= 0.092 and abs(r["Pr_G_R2"] - 0.96) <= 0.096 and
                           abs(r["YP_L_R2"] - 0.98) <= 0.098 and abs(r["YP_G_R2"] - 0.95) <= 0.095)
    r["order_Pr_G_gt_L"] = bool(r["Pr_G_R2"] > r["Pr_L_R2"])
    r["order_YP_L_gt_G"] = bool(r["YP_L_R2"] > r["YP_G_R2"])
    rows.append(r)
t5b = pd.DataFrame(rows)
print(t5b.round(4).to_string(index=False))
pb = t5b[t5b.group == "pooled"]
print("\npooled, every reading: within tolerance", bool(pb.within_tol.all()), "; Pr ordering", bool(pb.order_Pr_G_gt_L.all()),
      "; YP ordering", bool(pb.order_YP_L_gt_G.all()))

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
pos, my, pos2, mp = s2fits[("pooled", "a", "a")]
xx = np.linspace(-7, 7, 200)
for ax, (nm, x, y) in zip(axs, (("YP", pos, 100 * my / at(pos, my, 0)), ("Pr", pos2, 100 * mp / at(pos2, mp, 0)))):
    ax.plot(x, y, "ko", label=nm)
    for fn, fun, col in (("Lorentzian", cc.lorentz, "g"), ("Gaussian", cc.gauss, "r")):
        f = cc.fit(fun, x, y, 5)
        ax.plot(xx, fun(xx, f["a"], f["w"]), col, label=f"{fn} R2={f['R2']:.3f}")
    ax.set_title(f"{nm}, 0.2 MHz CANCAN wells with Pr, pooled (O1 a, O2 a)")
    ax.set_xlabel("distance (mm)")
    ax.set_ylabel("% of 0-mm value")
    ax.legend(fontsize=8)
    print(nm, "plotted range:", np.round(np.nanmin(y), 1), "to", np.round(np.nanmax(y), 1))
plt.tight_layout()
plt.show()

**Result: not reproduced.** For the pooled wells, the R2 values are 0.77 to 0.93 (Pr, Lorentzian), 0.74 to 0.89 (Pr, Gaussian), 0.68 to 0.88 (YP, Lorentzian) and 0.62 to 0.80 (YP, Gaussian): not all within tolerance under any reading, and for Pr the Lorentzian fits better than the Gaussian under every reading, the reverse of the paper. The YP ordering (Lorentzian better) holds. The x5 wells alone pass under one reading (O1 a, O2 b). This rests on the assumption that these wells had the electrodes touching the monolayer.

## fig7 claims (blocked): CT26 central YP by burst label

The Figure 7 claims need a CHO series over burst numbers, which the dataset does not hold, and the
dataset's burst labels cannot be matched to the paper's burst numbers from the files. The CT26 values
at 0.2 MHz by label are printed for the record; they decide no outcome.

In [ ]:
rows = []
for lab in ["x1", "x3", "x5"]:
    v = per_area(sel("0.2 MHz", "CANCAN", lab), "diag", "a", "a", xs=[0])
    rows.append({"label": lab, "n": len(v), "centre_YP_mean": v.mean(), "se": v.std(ddof=1) / np.sqrt(len(v))})
print(pd.DataFrame(rows).round(1).to_string(index=False))
print(f"\nnotebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| fig2-heating | Blocked: data missing (no film images) |
| fig3a-centre-enhanced | Not reproduced |
| fig3bc-rate-suppresses-off-target | Not reproduced |
| fig3-lorentzian-profile | Not reproduced |
| fig3-tail-deviation-falls-with-rate | Partially reproduced (burst label x1 only) |
| fig4b-centre-no-rate-effect | Reproduced |
| fig4b-near-electrode-decline | Not reproduced |
| fig4b-40-fold-ratio | Not reproduced |
| fig4c-fwhm-falls-with-rate | Blocked: data missing (no 1 kHz or 10 kHz) |
| fig4c-log-trend | Blocked: data missing (no 1 kHz or 10 kHz) |
| fig5-raised-electrodes-weaken-centre | Blocked: data missing (no elevations) |
| fig5ab-yp-spot-slightly-larger-than-pr | Not reproduced (assumes electrodes touching) |
| fig5b-gaussian-pr-lorentzian-yp | Not reproduced (assumes electrodes touching) |
| fig5c-nuclear-yp-centre | Blocked: data missing |
| fig6b, fig6c, fig6d | Blocked: data missing (600-ns pulses only) |
| fig7be, fig7cf, fig7-sharpest-one-burst | Blocked: data missing (no CHO burst series) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The array centre is found from the electrode footprints
in the YP image, or, in faint images, from the centroid of the stain; the paper placed ROIs by hand
in MetaMorph. Pixel sizes for the CT26 images are inferred from the array geometry. The paper's
burst number (2 bursts (x4)) cannot be identified among the dataset's labels, so the rate claims are
tested at the two labels both rates share, x1 and x5. Every rate claim can use only 1 Hz and
0.2 MHz.

What was found about the dataset. The protocol text in the dandiset and every file (3.8-mm electrode
spacing, 3.2-kV first pulse, 75/50/25% steps) does not describe the paper's array or packets, and
the CT26 images show the paper's 15.3-mm array. The burst number is given only as a subject label,
most files say "Unknown repetitions", and the CT26 files state no pixel size. The dataset holds no
images at 1 kHz or 10 kHz, no other pulse durations, no electrode elevations, no film images and no
CHO burst series; the CHO images (one burst label, 13 mm across) do not show the electrode array,
and they include a 1 Hz CHO set the paper does not report. At the labels the two rates share, the
0.2 MHz CANCAN wells show either almost no stain (x1) or a broad pattern that is not focused at the
centre (x5), unlike the paper's description of a sharp central spot with 40-fold suppression near
the electrodes, so the images in the dandiset may not be the experiments of the paper's figures.